# Bank Marketing: previsao de assinatura

Classificacao com Random Forest para prever clientes com maior chance de assinar um deposito a prazo.

In [ ]:
!pip install -q ucimlrepo joblib

In [ ]:
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from ucimlrepo import fetch_ucirepo

SEED = 42
REVOCACAO_MINIMA = 0.55
PRECISAO_DESEJADA = 0.40
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

## 1. Carregamento dos dados

In [ ]:
bank_marketing = fetch_ucirepo(id=222)
X = bank_marketing.data.features.copy()
y = bank_marketing.data.targets.copy()

X.columns = X.columns.str.strip()
y.columns = y.columns.str.strip()
y = y.squeeze().astype(str).str.strip().str.lower()

print(f'Linhas: {X.shape[0]} | Atributos: {X.shape[1]}')
display(X.head())
print(bank_marketing.metadata)
display(bank_marketing.variables)

## 2. Análise exploratória

In [ ]:
X_eda = X.replace('?', np.nan).copy()

print('Valores ausentes:')
display(X_eda.isna().sum().sort_values(ascending=False).to_frame('quantidade'))
print('Duplicidades nas features:', int(X_eda.duplicated().sum()))
print('Distribuicao do alvo:')
display(y.value_counts().rename(index={'no': 'Nao', 'yes': 'Sim'}).to_frame('quantidade'))
display(y.value_counts(normalize=True).mul(100).round(2).rename(index={'no': 'Nao', 'yes': 'Sim'}).to_frame('percentual'))

plt.figure(figsize=(6, 4))
sns.countplot(x=y.map({'no': 'Nao', 'yes': 'Sim'}))
plt.title('Distribuicao do alvo')
plt.xlabel('Assinou deposito a prazo')
plt.ylabel('Quantidade')
plt.show()

## 3. Limpeza e definição do alvo

A variavel `duration` é excluída porque a duração só é conhecida durante ou depois da ligação.

In [ ]:
REMOVER_DURATION_POR_VAZAMENTO = True
X = X.replace('?', np.nan).copy()

for coluna in ['age', 'balance', 'day', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous']:
    if coluna in X.columns:
        X[coluna] = pd.to_numeric(X[coluna], errors='coerce')

dataset = X.copy()
dataset['y'] = y.to_numpy()
duplicatas = int(dataset.duplicated().sum())
dataset = dataset.drop_duplicates().reset_index(drop=True)

X = dataset.drop(columns='y')
y = dataset['y'].map({'no': 0, 'yes': 1}).astype(int)

if REMOVER_DURATION_POR_VAZAMENTO and 'duration' in X.columns:
    X = X.drop(columns='duration')

print(f'Duplicatas completas removidas: {duplicatas}')
print(f'Formato final das features: {X.shape}')
print('duration removida:', REMOVER_DURATION_POR_VAZAMENTO)

## 4. Divisao dos dados

Treino: 80%. Teste: 20%. A estratificacao mantem a proporcao de assinantes.

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

print(f'Treino: {len(X_treino)} linhas')
print(f'Teste: {len(X_teste)} linhas')
print(f'Taxa positiva no treino: {y_treino.mean():.3f}')
print(f'Taxa positiva no teste: {y_teste.mean():.3f}')

## 5. Treinamento e importancia das variaveis

O Random Forest usa parametros fixos. A importancia mostra as variaveis mais usadas pelo conjunto de arvores.

In [ ]:
def criar_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

colunas_numericas = X_treino.select_dtypes(include='number').columns.tolist()
colunas_categoricas = X_treino.select_dtypes(exclude='number').columns.tolist()

preprocessamento_arvore = ColumnTransformer(transformers=[
    ('numericas', SimpleImputer(strategy='median'), colunas_numericas),
    ('categoricas', Pipeline(steps=[
        ('imputacao', SimpleImputer(strategy='most_frequent')),
        ('codificacao', criar_one_hot_encoder()),
    ]), colunas_categoricas),
])

PARAMETROS_RF = {
    'n_estimators': 200,
    'criterion': 'entropy',
    'max_depth': 12,
    'min_samples_leaf': 3,
    'max_features': 'sqrt',
    'class_weight': 'balanced',
}

modelo_rf = Pipeline(steps=[
    ('preprocessamento', preprocessamento_arvore),
    ('random_forest', RandomForestClassifier(
        **PARAMETROS_RF,
        random_state=SEED,
        n_jobs=-1,
    )),
])

modelo_rf.fit(X_treino, y_treino)
print('Random Forest treinado.')
print(PARAMETROS_RF)

nomes_features = modelo_rf.named_steps['preprocessamento'].get_feature_names_out()
importancias = pd.Series(
    modelo_rf.named_steps['random_forest'].feature_importances_,
    index=nomes_features,
).sort_values(ascending=False).head(15)

display(importancias.to_frame('importancia'))

plt.figure(figsize=(8, 5))
importancias.sort_values().plot(kind='barh', color='#11665B')
plt.title('15 variaveis com maior importancia')
plt.xlabel('Importancia')
plt.tight_layout()
plt.show()

## 6. Previsoes e metricas

O modelo usa o metodo predict. A decisao segue o limiar padrao do Random Forest.

In [ ]:
previsoes_treino_rf = modelo_rf.predict(X_treino)
previsoes_teste_rf = modelo_rf.predict(X_teste)
probabilidades_treino_rf = modelo_rf.predict_proba(X_treino)[:, 1]
probabilidades_teste_rf = modelo_rf.predict_proba(X_teste)[:, 1]

def calcular_metricas(nome, y_real, previsoes, probabilidades):
    return {
        'conjunto': nome,
        'acuracia': accuracy_score(y_real, previsoes),
        'precisao': precision_score(y_real, previsoes, zero_division=0),
        'revocacao': recall_score(y_real, previsoes, zero_division=0),
        'f1': f1_score(y_real, previsoes, zero_division=0),
        'auc_roc': roc_auc_score(y_real, probabilidades),
    }

metricas_rf_tabela = pd.DataFrame([
    calcular_metricas('Treino', y_treino, previsoes_treino_rf, probabilidades_treino_rf),
    calcular_metricas('Teste', y_teste, previsoes_teste_rf, probabilidades_teste_rf),
]).set_index('conjunto')

display(metricas_rf_tabela.style.format('{:.4f}'))

print('Relatorio de classificacao no teste:')
print(classification_report(
    y_teste,
    previsoes_teste_rf,
    target_names=['Nao assinou', 'Assinou'],
    zero_division=0,
))

## 7. Matriz de confusao e curva ROC

In [ ]:
matriz_rf = confusion_matrix(y_teste, previsoes_teste_rf)
fpr_rf, tpr_rf, _ = roc_curve(y_teste, probabilidades_teste_rf)
auc_teste = roc_auc_score(y_teste, probabilidades_teste_rf)

fig, eixos = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(
    matriz_rf, annot=True, fmt='d', cmap='Greens', cbar=False,
    xticklabels=['Nao assinou', 'Assinou'],
    yticklabels=['Nao assinou', 'Assinou'], ax=eixos[0]
)
eixos[0].set_title('Matriz de confusao')
eixos[0].set_xlabel('Previsao')
eixos[0].set_ylabel('Verdadeiro')

eixos[1].plot(fpr_rf, tpr_rf, label=f'AUC = {auc_teste:.3f}')
eixos[1].plot([0, 1], [0, 1], '--', color='gray')
eixos[1].set_title('Curva ROC')
eixos[1].set_xlabel('Taxa de falsos positivos')
eixos[1].set_ylabel('Taxa de verdadeiros positivos')
eixos[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

## 8. Clientes ficticios

Os cinco perfis demonstram o uso do modelo. As probabilidades sao estimativas do classificador.

In [ ]:
clientes_ficticios = pd.DataFrame({
    'age': [28, 54, 42, 67, 35],
    'job': ['student', 'management', 'blue-collar', 'retired', 'entrepreneur'],
    'marital': ['single', 'married', 'married', 'divorced', 'single'],
    'education': ['tertiary', 'tertiary', 'secondary', 'primary', 'tertiary'],
    'default': ['no', 'no', 'no', 'no', 'yes'],
    'balance': [1800, 3200, -250, 8500, -50],
    'housing': ['no', 'yes', 'yes', 'no', 'no'],
    'loan': ['no', 'no', 'yes', 'no', 'yes'],
    'contact': ['cellular', 'cellular', 'telephone', 'telephone', 'cellular'],
    'day_of_week': [5, 14, 20, 3, 27],
    'month': ['may', 'aug', 'nov', 'mar', 'nov'],
    'campaign': [1, 1, 3, 1, 4],
    'pdays': [-1, 120, -1, 90, 200],
    'previous': [0, 2, 0, 1, 3],
    'poutcome': [np.nan, 'success', np.nan, 'failure', 'other'],
}, index=['Cliente 1', 'Cliente 2', 'Cliente 3', 'Cliente 4', 'Cliente 5'])

clientes_ficticios = clientes_ficticios.reindex(columns=X.columns)
probabilidades_ficticios_rf = modelo_rf.predict_proba(clientes_ficticios)[:, 1]
previsoes_ficticios_rf = modelo_rf.predict(clientes_ficticios)

resultado_ficticios_rf = pd.DataFrame({
    'cliente': clientes_ficticios.index,
    'probabilidade_assinatura': probabilidades_ficticios_rf.round(4),
    'previsao': np.where(previsoes_ficticios_rf == 1, 'Assinaria', 'Nao assinaria'),
})
display(resultado_ficticios_rf)